# 🏥 Medical NLP Symptom Extraction Pipeline
### Production-Grade | 10-Layer Architecture | CustomTkinter GUI

---

**Architecture Overview:**
```
User Input → Spell Correct → Noise Filter → Dict Match → Semantic Match
           → Fuzzy Fallback → Negation → Conflict → Boost → Score → Output
```

| Layer | Method | Confidence Weight |
|-------|--------|------------------|
| 1 | Dictionary (exact) | 1.00 |
| 2 | Semantic (spaCy) | 0.85 × sim |
| 3 | Fuzzy fallback | 0.70 × score |
| + | Context boost | additive |

> **Requires:** `en_core_web_md` spaCy model — run `python -m spacy download en_core_web_md`

## Cell 1 — Install Dependencies

In [8]:
# ─────────────────────────────────────────────
#  CELL 1 · Install & Download
# ─────────────────────────────────────────────
import subprocess, sys

packages = [
    "customtkinter",
    "pyspellchecker",
    "rapidfuzz",
    "scikit-learn",
    "pandas",
    "numpy",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Download spaCy medium model (needed for word vectors)
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_md", "-q"])

print("✅  All dependencies installed.")

✅  All dependencies installed.


## Cell 2 — Imports & Global Configuration

In [9]:
# ─────────────────────────────────────────────
#  CELL 2 · Imports & Global Config
# ─────────────────────────────────────────────
import re
import numpy as np
import pandas as pd
import spacy
import threading
import tkinter as tk

from collections import defaultdict
from spellchecker import SpellChecker
from rapidfuzz import process, fuzz
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

import customtkinter as ctk

# ── spaCy ────────────────────────────────────
nlp = spacy.load("en_core_web_md")

# ── CustomTkinter theme ──────────────────────
ctk.set_appearance_mode("dark")        # "dark" | "light" | "system"
ctk.set_default_color_theme("blue")   # "blue" | "green" | "dark-blue"

print("✅  Imports complete.")

✅  Imports complete.


## Cell 3 — Medical Domain Vocabulary & Maps

In [10]:
# ─────────────────────────────────────────────
#  CELL 3 · Medical Vocabulary & Domain Maps
# ─────────────────────────────────────────────

# Protected medical terms (never spell-corrected)
MEDICAL_DOMAIN_WORDS = {
    "nausea", "vomit", "vomiting", "diarrhea", "diarrhoea", "dyspnea",
    "palpitation", "palpitations", "fatigue", "malaise", "myalgia",
    "arthralgia", "hemoptysis", "dysphagia", "syncope", "vertigo",
    "tinnitus", "pruritus", "urticaria", "edema", "jaundice", "cyanosis",
    "bradycardia", "tachycardia", "rhinorrhea", "epistaxis", "diplopia",
    "dyspepsia", "haematuria", "haemoptysis", "paraesthesia", "pyrexia",
}

# Phrase → dataset column name  (extend with your full dataset columns)
MEDICAL_MAP = {
    # ── Fever / Temperature ──────────────────
    "high fever":                 "high_fever",
    "mild fever":                 "mild_fever",
    "fever":                      "high_fever",
    "temperature":                "high_fever",
    "pyrexia":                    "high_fever",
    # ── Pain ────────────────────────────────
    "chest pain":                 "chest_pain",
    "abdominal pain":             "abdominal_pain",
    "stomach pain":               "stomach_pain",
    "belly pain":                 "abdominal_pain",
    "joint pain":                 "joint_pain",
    "muscle pain":                "muscle_pain",
    "muscle ache":                "muscle_pain",
    "back pain":                  "back_pain",
    "headache":                   "headache",
    "head ache":                  "headache",
    "migraine":                   "headache",
    "neck pain":                  "neck_pain",
    # ── Respiratory ──────────────────────────
    "shortness of breath":        "breathlessness",
    "difficulty breathing":       "breathlessness",
    "breathlessness":             "breathlessness",
    "cough":                      "cough",
    "dry cough":                  "cough",
    "continuous sneezing":        "continuous_sneezing",
    "sneezing":                   "continuous_sneezing",
    "runny nose":                 "runny_nose",
    "nasal congestion":           "congestion",
    "congestion":                 "congestion",
    # ── GI ───────────────────────────────────
    "nausea":                     "nausea",
    "vomiting":                   "vomiting",
    "vomit":                      "vomiting",
    "diarrhoea":                  "diarrhoea",
    "diarrhea":                   "diarrhoea",
    "loose stools":               "diarrhoea",
    "constipation":               "constipation",
    "acidity":                    "acidity",
    "heartburn":                  "acidity",
    "indigestion":                "indigestion",
    "loss of appetite":           "loss_of_appetite",
    "no appetite":                "loss_of_appetite",
    "bloating":                   "abdominal_pain",
    # ── Skin ────────────────────────────────
    "itching":                    "itching",
    "skin rash":                  "skin_rash",
    "rash":                       "skin_rash",
    "nodal skin eruptions":       "nodal_skin_eruptions",
    "yellowish skin":             "yellowing_of_eyes",
    "jaundice":                   "yellowing_of_eyes",
    "yellow skin":                "yellowing_of_eyes",
    # ── Fatigue / General ────────────────────
    "fatigue":                    "fatigue",
    "tiredness":                  "fatigue",
    "exhaustion":                 "fatigue",
    "weakness":                   "weakness_in_limbs",
    "lethargy":                   "lethargy",
    "malaise":                    "malaise",
    "dizziness":                  "dizziness",
    "lightheaded":                "dizziness",
    "chills":                     "chills",
    "shivering":                  "shivering",
    "sweating":                   "sweating",
    "night sweats":               "sweating",
    # ── Neuro ────────────────────────────────
    "anxiety":                    "anxiety",
    "depression":                 "depression",
    "confusion":                  "lack_of_concentration",
    "loss of concentration":      "lack_of_concentration",
    "memory loss":                "lack_of_concentration",
    "blurred vision":             "blurred_and_distorted_vision",
    "distorted vision":           "blurred_and_distorted_vision",
    # ── Weight ───────────────────────────────
    "weight loss":                "weight_loss",
    "losing weight":              "weight_loss",
    "weight gain":                "weight_gain",
    # ── Other ────────────────────────────────
    "swollen lymph nodes":        "swelled_lymph_nodes",
    "swollen glands":             "swelled_lymph_nodes",
    "dehydration":                "dehydration",
    "excessive thirst":           "excessive_hunger",
    "frequent urination":         "frequent_urination",
    "irregular sugar level":      "irregular_sugar_level",
    "palpitations":               "palpitations",
    "fast heartbeat":             "fast_heart_rate",
    "slow heartbeat":             "slow_heart_rate",
    "swelling":                   "swelling_of_stomach",
    "stiff neck":                 "stiff_neck",
    "throat pain":                "throat_pain",
    "sore throat":                "throat_pain",
    "burning urination":          "burning_urination",
    "painful urination":          "burning_urination",
}

# Inputs that signal the user has no symptoms
REJECTION_PHRASES = {
    "i am fine", "i feel fine", "i feel okay", "i feel ok",
    "nothing wrong", "feeling good", "feeling great", "all good",
    "im fine", "im okay", "im ok", "no complaints", "i am well",
    "everything is fine", "i am healthy", "feeling better",
    "i am good", "good", "fine", "okay", "ok", "nothing",
}

# Negation vocabulary
NEGATION_TOKENS = {
    "no", "not", "never", "without", "deny", "denies", "denying",
    "absent", "negative", "none", "free", "lack", "lacking",
    "dont", "doesn't", "didn't", "haven't", "hasn't", "isn't",
}

# Mutually exclusive symptom pairs
CONFLICT_MAP = {
    "chills":           {"absence_of_chills"},
    "absence_of_chills":{"chills"},
    "constipation":     {"diarrhoea"},
    "diarrhoea":        {"constipation"},
    "weight_gain":      {"weight_loss"},
    "weight_loss":      {"weight_gain"},
    "fast_heart_rate":  {"slow_heart_rate"},
    "slow_heart_rate":  {"fast_heart_rate"},
}

# Pipeline constants
CONFIDENCE_THRESHOLD = 0.50
FUZZY_THRESHOLD      = 82
NEGATION_WINDOW      = 5
BOOST_FACTOR         = 0.15
BOOST_DECAY          = 0.50
MIN_CONTENT_TOKENS   = 2

print(f"✅  MEDICAL_MAP loaded: {len(MEDICAL_MAP)} phrase entries.")

✅  MEDICAL_MAP loaded: 84 phrase entries.


## Cell 4 — MedicalEngine Class (Full 10-Layer Pipeline)

In [11]:
# ─────────────────────────────────────────────
#  CELL 4 · MedicalEngine — 10-Layer Pipeline
# ─────────────────────────────────────────────

class MedicalEngine:
    """
    Production-grade symptom extraction + disease prediction engine.

    Pipeline:
      1  Spelling correction   (medical-term-aware)
      2  Noise filter          (reject non-medical inputs)
      3  Layer 1 match         (dictionary, longest-first + masking)
      4  Layer 2 match         (spaCy semantic similarity)
      5  Layer 3 match         (fuzzy fallback via rapidfuzz)
      6  Negation detection    (dep parse + sentence boundaries)
      7  Conflict resolution   (remove contradictory pairs)
      8  Context boosting      (co-occurrence graph)
      9  Confidence scoring    (weighted multi-layer evidence)
     10  Threshold gate        (only confident symptoms pass)
    """

    # ─── Initialisation ──────────────────────────────────────────────────

    def __init__(self, csv_path: str):
        print("⚙  Loading dataset …")
        df = pd.read_csv(csv_path)
        df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

        self.le = LabelEncoder()
        df["prognosis"] = self.le.fit_transform(df["prognosis"])
        self.X = df.drop("prognosis", axis=1)
        self.y = df["prognosis"]

        print("⚙  Training RandomForest …")
        self.model = RandomForestClassifier(n_estimators=250, random_state=42)
        self.model.fit(self.X, self.y)

        self.symptoms_list  = list(self.X.columns)
        self.clean_symptoms = [s.replace("_", " ") for s in self.symptoms_list]

        # SpellChecker with protected vocabulary
        self.spell = SpellChecker()
        self._protected_vocab = MEDICAL_DOMAIN_WORDS.copy()
        for sym in self.symptoms_list:
            for token in sym.split("_"):
                self._protected_vocab.add(token.lower())

        print("⚙  Building co-occurrence graph …")
        self.cooccurrence_graph = self._build_cooccurrence_graph()

        print("✅  MedicalEngine ready.")

    # ─── Step 1 · Spelling Correction ────────────────────────────────────

    def correct_spelling(self, text: str) -> str:
        """
        Corrects misspelled words while protecting:
          • medical vocabulary
          • 3-char-or-shorter tokens
          • ALL-CAPS abbreviations (UTI, IBS …)
          • numeric tokens
        """
        words     = text.split()
        corrected = []
        for word in words:
            clean = re.sub(r"[^a-zA-Z]", "", word).lower()
            if (
                len(clean) <= 3
                or clean in self._protected_vocab
                or word.isupper()
                or word.isdigit()
            ):
                corrected.append(word)
            else:
                suggestion = self.spell.correction(clean)
                corrected.append(suggestion if suggestion else word)
        return " ".join(corrected)

    # ─── Step 2 · Noise Filter ────────────────────────────────────────────

    def is_noise(self, text: str) -> bool:
        """
        Three-gate filter:
          Gate 1 – exact rejection phrase match
          Gate 2 – fewer than MIN_CONTENT_TOKENS meaningful tokens
          Gate 3 – max semantic similarity to any symptom below 0.35
        """
        text_lower = text.lower().strip()
        stripped   = re.sub(r"[^a-z\s]", "", text_lower).strip()

        # Gate 1
        if text_lower in REJECTION_PHRASES or stripped in REJECTION_PHRASES:
            return True

        # Gate 2
        doc = nlp(text_lower)
        content_tokens = [
            t for t in doc
            if not t.is_stop and not t.is_punct and len(t.text) > 2
        ]
        if len(content_tokens) < MIN_CONTENT_TOKENS:
            return True

        # Gate 3 – sample first 40 symptoms for speed
        query_doc = nlp(" ".join(t.text for t in content_tokens))
        if query_doc.has_vector:
            sample   = [nlp(s) for s in self.clean_symptoms[:40]]
            max_sim  = max(
                (query_doc.similarity(sd) for sd in sample if sd.has_vector),
                default=0.0,
            )
            if max_sim < 0.35:
                return True

        return False

    # ─── Step 3 · Layer 1 – Dictionary Match ─────────────────────────────

    def _layer1_dictionary_match(
        self, doc, masked_indices: set
    ) -> list:
        """
        Exact phrase match (longest first).  Returns list of
        (symptom_col, confidence, span_start, span_end).
        Updates masked_indices in-place.
        """
        results     = []
        sorted_keys = sorted(MEDICAL_MAP.keys(), key=lambda k: len(k.split()), reverse=True)

        for key in sorted_keys:
            key_tokens = key.lower().split()
            key_len    = len(key_tokens)

            for i in range(len(doc) - key_len + 1):
                window = [doc[i + j].text.lower() for j in range(key_len)]
                if window != key_tokens:
                    continue
                if any((i + j) in masked_indices for j in range(key_len)):
                    continue

                symptom = MEDICAL_MAP[key]
                results.append((symptom, 1.0, i, i + key_len))

                for j in range(key_len):
                    masked_indices.add(i + j)
                break

        return results

    # ─── Step 4 · Layer 2 – Semantic Match ───────────────────────────────

    def _layer2_semantic_match(
        self, doc, masked_indices: set, top_n: int = 3
    ) -> list:
        """
        Cosine similarity on remainder (unmasked, non-stop) tokens.
        Returns up to top_n (symptom, similarity_score) tuples.
        """
        remainder = [
            t for i, t in enumerate(doc)
            if i not in masked_indices and not t.is_stop and len(t.text) > 2
        ]
        if not remainder:
            return []

        query_vec = nlp(" ".join(t.text for t in remainder))
        if not query_vec.has_vector:
            return []

        candidates = []
        for idx, sym_text in enumerate(self.clean_symptoms):
            sym_vec = nlp(sym_text)
            if not sym_vec.has_vector:
                continue
            sim = query_vec.similarity(sym_vec)
            if sim > 0.75:
                candidates.append((self.symptoms_list[idx], sim))

        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates[:top_n]

    # ─── Step 5 · Layer 3 – Fuzzy Fallback ───────────────────────────────

    def _layer3_fuzzy_fallback(
        self, doc, masked_indices: set
    ) -> list:
        """
        token_set_ratio fuzzy match on remainder tokens.
        Only called when Layer 2 is below threshold.
        Returns at most one (symptom, normalised_score).
        """
        remainder = " ".join(
            t.text for i, t in enumerate(doc)
            if i not in masked_indices and not t.is_stop and len(t.text) > 2
        )
        if len(remainder) < 4:
            return []

        match, score, _ = process.extractOne(
            remainder, self.clean_symptoms, scorer=fuzz.token_set_ratio
        )
        if score >= FUZZY_THRESHOLD:
            sym_col = self.symptoms_list[self.clean_symptoms.index(match)]
            return [(sym_col, (score / 100.0) * 0.75)]
        return []

    # ─── Step 6 · Negation Detection ─────────────────────────────────────

    def _is_negated(self, doc, span_start: int, span_end: int) -> bool:
        """
        Sentence-scoped negation using:
          • dependency arcs (neg relation)
          • head-chain walk (3 levels)
          • pre-span window within same sentence
        """
        span_sent = next(
            (s for s in doc.sents if s.start <= span_start < s.end), None
        )
        if span_sent is None:
            return False
        sent_start = span_sent.start

        for i in range(span_start, span_end):
            token = doc[i]
            for child in token.children:
                if child.dep_ == "neg" or child.text.lower() in NEGATION_TOKENS:
                    return True
            head = token.head
            for _ in range(3):
                if head.dep_ == "neg" or head.text.lower() in NEGATION_TOKENS:
                    return True
                if head == head.head:
                    break
                head = head.head

        window_start = max(sent_start, span_start - NEGATION_WINDOW)
        for i in range(window_start, span_start):
            if doc[i].text.lower() in NEGATION_TOKENS:
                return True
        return False

    def _is_negated_heuristic(self, doc, symptom: str) -> bool:
        """
        Lightweight negation check for symptoms without explicit token spans
        (Layer 2 & 3 detections).
        """
        sym_words = set(symptom.replace("_", " ").lower().split())
        for i, token in enumerate(doc):
            if token.text.lower() in sym_words:
                window_start = max(0, i - NEGATION_WINDOW)
                for j in range(window_start, i):
                    if doc[j].text.lower() in NEGATION_TOKENS:
                        return True
        return False

    # ─── Step 7 · Conflict Resolution ────────────────────────────────────

    def _resolve_conflicts(self, scored: dict) -> dict:
        """
        Removes the lower-confidence member of any contradictory pair.
        """
        to_remove = set()
        for sym, conf in scored.items():
            if sym in CONFLICT_MAP:
                for conflict in CONFLICT_MAP[sym]:
                    if conflict in scored:
                        victim = conflict if conf >= scored[conflict] else sym
                        to_remove.add(victim)
        return {k: v for k, v in scored.items() if k not in to_remove}

    # ─── Step 8 · Context Boosting ────────────────────────────────────────

    def _build_cooccurrence_graph(self) -> dict:
        """
        Builds {symptom: [(related_sym, weight), …]} from training data.
        Built once in __init__.
        """
        co_counts = defaultdict(lambda: defaultdict(int))
        for _, row in self.X.iterrows():
            present = [col for col in self.symptoms_list if row[col] == 1]
            for i, a in enumerate(present):
                for b in present[i + 1:]:
                    co_counts[a][b] += 1
                    co_counts[b][a] += 1
        graph = {}
        for sym, related in co_counts.items():
            total = sum(related.values())
            if total == 0:
                continue
            ranked = sorted(
                ((rel, cnt / total) for rel, cnt in related.items()),
                key=lambda x: x[1], reverse=True,
            )
            graph[sym] = ranked[:10]
        return graph

    def _apply_context_boost(self, scored: dict) -> dict:
        """
        Boosts candidates already in the pool based on co-occurrence.
        Never adds new symptoms.
        """
        boosted   = dict(scored)
        confirmed = list(scored.keys())
        for sym in confirmed:
            if sym not in self.cooccurrence_graph:
                continue
            decay = 1.0
            for related, weight in self.cooccurrence_graph[sym]:
                if related in boosted:
                    boost           = BOOST_FACTOR * weight * decay
                    boosted[related] = min(1.0, boosted[related] + boost)
                    decay          *= BOOST_DECAY
        return boosted

    # ─── Step 9 · Confidence Scoring ──────────────────────────────────────

    def _compute_scores(
        self,
        layer1: list,
        layer2: list,
        layer3: list,
    ) -> dict:
        """
        Combines evidence from all layers:
          base = max single-layer score
          bonus = +0.05 per additional layer that agrees
        """
        WEIGHTS     = {"layer1": 1.00, "layer2": 0.85, "layer3": 0.70}
        scores      = defaultdict(list)
        layer_count = defaultdict(int)

        for sym, conf, *_ in layer1:
            scores[sym].append(conf * WEIGHTS["layer1"])
            layer_count[sym] += 1
        for sym, sim in layer2:
            scores[sym].append(sim * WEIGHTS["layer2"])
            layer_count[sym] += 1
        for sym, norm in layer3:
            scores[sym].append(norm * WEIGHTS["layer3"])
            layer_count[sym] += 1

        final = {}
        for sym, evidence in scores.items():
            base  = max(evidence)
            bonus = (layer_count[sym] - 1) * 0.05
            final[sym] = min(1.0, base + bonus)
        return final

    # ─── Step 10 · Master Extract (orchestrates all 9 steps) ──────────────

    def extract_symptoms(
        self, text: str, return_scores: bool = False
    ):
        """
        Full 10-layer pipeline entry point.

        Returns:
          list[str]  — confirmed symptom column names, OR
          (list[str], dict[str, float])  when return_scores=True
        """
        empty = ([], {}) if return_scores else []

        text_lower = text.lower().strip()
        if not text_lower:
            return empty

        # Step 2 – noise
        if self.is_noise(text_lower):
            return empty

        # Step 1 – spelling
        corrected = self.correct_spelling(text_lower)
        doc       = nlp(corrected)
        masked    = set()

        # Step 3 – dict match
        l1 = self._layer1_dictionary_match(doc, masked)

        # Step 4 – semantic
        l2 = self._layer2_semantic_match(doc, masked)

        # Step 5 – fuzzy (only when semantic is weak)
        best_sem = max((s for _, s in l2), default=0.0)
        l3       = self._layer3_fuzzy_fallback(doc, masked) if best_sem < 0.80 else []

        # Step 9 – score
        scored = self._compute_scores(l1, l2, l3)

        # Step 6 – negation
        negated = {
            sym for sym, _, s, e in l1
            if self._is_negated(doc, s, e)
        }
        for sym, _ in (l2 + l3):
            if self._is_negated_heuristic(doc, sym):
                negated.add(sym)
        scored = {k: v for k, v in scored.items() if k not in negated}

        # Step 7 – conflicts
        scored = self._resolve_conflicts(scored)

        # Step 8 – boost
        scored = self._apply_context_boost(scored)

        # Step 10 – threshold gate
        confirmed = [sym for sym, conf in scored.items() if conf >= CONFIDENCE_THRESHOLD]

        return (confirmed, {k: round(v, 3) for k, v in scored.items()}) \
            if return_scores else confirmed

    # ─── Recommendations & Prediction ────────────────────────────────────

    def get_recommendations(self, current_symptoms: list, top_n: int = 4) -> list:
        """Returns symptom names likely to co-occur with the current set."""
        if not current_symptoms:
            return []

        mask = np.ones(len(self.X), dtype=bool)
        for s in current_symptoms:
            if s in self.symptoms_list:
                mask &= (self.X[s] == 1)

        subset = self.X[mask]
        if len(subset) == 0:
            mask = np.zeros(len(self.X), dtype=bool)
            for s in current_symptoms:
                if s in self.symptoms_list:
                    mask |= (self.X[s] == 1)
            subset = self.X[mask]

        if len(subset) == 0:
            return []

        counts      = subset.sum(axis=0).drop(labels=current_symptoms, errors="ignore")
        top_syms    = counts.nlargest(top_n).index.tolist()
        return [s.replace("_", " ") for s in top_syms if counts[s] > 0]

    def predict_top_two(self, symptoms: list) -> list:
        """Returns top-2 (disease_name, confidence_pct) predictions."""
        if not symptoms:
            return []
        vector = np.zeros(len(self.symptoms_list))
        for s in symptoms:
            if s in self.symptoms_list:
                vector[self.symptoms_list.index(s)] = 1
        prob     = self.model.predict_proba([vector])[0]
        top2_idx = np.argsort(prob)[-2:][::-1]
        return [
            (self.le.inverse_transform([i])[0], round(prob[i] * 100, 1))
            for i in top2_idx
        ]

    def run_system_test(self, test_csv: str):
        """Validates model accuracy on a held-out test CSV."""
        from sklearn.metrics import accuracy_score
        df      = pd.read_csv(test_csv)
        df      = df.loc[:, ~df.columns.str.contains("^Unnamed")]
        X_test  = df.drop("prognosis", axis=1)
        y_true  = self.le.transform(df["prognosis"])
        y_pred  = self.model.predict(X_test)
        acc     = accuracy_score(y_true, y_pred)
        print(f"✅  Test accuracy: {acc * 100:.2f}%")
        return acc


print("✅  MedicalEngine class defined.")

✅  MedicalEngine class defined.


## Cell 5 — CustomTkinter GUI Application

In [12]:
# ─────────────────────────────────────────────
#  CELL 5 · CustomTkinter GUI
# ─────────────────────────────────────────────

class MedicalGUI(ctk.CTk):
    """
    Dark-themed medical symptom chatbot GUI built with CustomTkinter.

    Layout
    ┌─────────────────────────────────────────────────┐
    │  HEADER  (title + subtitle)                     │
    ├────────────────────────┬────────────────────────┤
    │  LEFT PANEL            │  RIGHT PANEL           │
    │  ● Chat transcript     │  ● Confirmed symptoms  │
    │  ● Input box           │  ● Predictions         │
    │  ● Send / Clear btns   │  ● Suggestions         │
    │                        │  ● Confidence bars     │
    └────────────────────────┴────────────────────────┘
    """

    # ── Palette ──────────────────────────────────────────────────────────
    COLORS = {
        "bg_primary":    "#0D1117",
        "bg_secondary":  "#161B22",
        "bg_card":       "#1C2128",
        "accent_blue":   "#2F81F7",
        "accent_teal":   "#3FB950",
        "accent_amber":  "#D29922",
        "accent_red":    "#F85149",
        "text_primary":  "#E6EDF3",
        "text_secondary":"#8B949E",
        "border":        "#30363D",
        "user_bubble":   "#1F4B8C",
        "bot_bubble":    "#1C2128",
        "tag_sym":       "#0D3349",
        "tag_pred1":     "#1A3A2A",
        "tag_pred2":     "#3A2A0A",
    }

    def __init__(self, engine: MedicalEngine):
        super().__init__()
        self.engine          = engine
        self.confirmed_syms  = []          # accumulated confirmed symptoms
        self.conv_history    = []          # (role, text) for display

        self._configure_window()
        self._build_header()
        self._build_body()
        self._build_footer()
        self._post_bot_message(
            "Hello! I'm your medical assistant. "
            "Please describe your symptoms in natural language — "
            "for example: \"I have a high fever and sore throat.\" "
            "I'll identify the symptoms and suggest possible conditions."
        )

    # ── Window Setup ─────────────────────────────────────────────────────

    def _configure_window(self):
        self.title("MedAssist — Symptom Analysis")
        self.geometry("1200x780")
        self.minsize(900, 600)
        self.configure(fg_color=self.COLORS["bg_primary"])
        self.grid_rowconfigure(1, weight=1)
        self.grid_columnconfigure(0, weight=1)

    # ── Header ───────────────────────────────────────────────────────────

    def _build_header(self):
        hdr = ctk.CTkFrame(
            self, fg_color=self.COLORS["bg_secondary"],
            corner_radius=0, height=70,
        )
        hdr.grid(row=0, column=0, sticky="ew", padx=0, pady=0)
        hdr.grid_propagate(False)

        ctk.CTkLabel(
            hdr,
            text="🏥  MedAssist",
            font=ctk.CTkFont(family="Helvetica", size=26, weight="bold"),
            text_color=self.COLORS["text_primary"],
        ).place(relx=0.04, rely=0.3)

        ctk.CTkLabel(
            hdr,
            text="10-layer NLP symptom extraction  ·  RandomForest prediction",
            font=ctk.CTkFont(size=12),
            text_color=self.COLORS["text_secondary"],
        ).place(relx=0.04, rely=0.65)

        # Status dot
        ctk.CTkLabel(
            hdr,
            text="● ONLINE",
            font=ctk.CTkFont(size=11, weight="bold"),
            text_color=self.COLORS["accent_teal"],
        ).place(relx=0.88, rely=0.4)

    # ── Body ─────────────────────────────────────────────────────────────

    def _build_body(self):
        body = ctk.CTkFrame(self, fg_color=self.COLORS["bg_primary"], corner_radius=0)
        body.grid(row=1, column=0, sticky="nsew", padx=0, pady=0)
        body.grid_rowconfigure(0, weight=1)
        body.grid_columnconfigure(0, weight=3)
        body.grid_columnconfigure(1, weight=2)

        self._build_chat_panel(body)
        self._build_info_panel(body)

    # ── Chat Panel (left) ─────────────────────────────────────────────────

    def _build_chat_panel(self, parent):
        frame = ctk.CTkFrame(
            parent, fg_color=self.COLORS["bg_secondary"],
            corner_radius=0,
        )
        frame.grid(row=0, column=0, sticky="nsew", padx=(0, 1), pady=0)
        frame.grid_rowconfigure(0, weight=1)
        frame.grid_columnconfigure(0, weight=1)

        # Scrollable transcript
        self.chat_scroll = ctk.CTkScrollableFrame(
            frame,
            fg_color=self.COLORS["bg_secondary"],
            scrollbar_button_color=self.COLORS["border"],
            corner_radius=0,
        )
        self.chat_scroll.grid(row=0, column=0, sticky="nsew", padx=0, pady=0)
        self.chat_scroll.grid_columnconfigure(0, weight=1)

        # Input area
        input_frame = ctk.CTkFrame(
            frame, fg_color=self.COLORS["bg_card"],
            corner_radius=0, height=110,
        )
        input_frame.grid(row=1, column=0, sticky="ew", padx=0, pady=0)
        input_frame.grid_propagate(False)
        input_frame.grid_columnconfigure(0, weight=1)

        self.input_box = ctk.CTkTextbox(
            input_frame,
            height=60,
            font=ctk.CTkFont(size=14),
            fg_color=self.COLORS["bg_secondary"],
            text_color=self.COLORS["text_primary"],
            border_color=self.COLORS["border"],
            border_width=1,
            corner_radius=8,
            wrap="word",
        )
        self.input_box.grid(row=0, column=0, padx=12, pady=(8, 4), sticky="ew")
        self.input_box.bind("<Return>", self._on_enter)
        self.input_box.bind("<Shift-Return>", lambda e: None)  # allow newline

        btn_frame = ctk.CTkFrame(input_frame, fg_color="transparent")
        btn_frame.grid(row=1, column=0, padx=12, pady=(0, 8), sticky="e")

        self.send_btn = ctk.CTkButton(
            btn_frame,
            text="Send  ↵",
            width=110, height=34,
            font=ctk.CTkFont(size=13, weight="bold"),
            fg_color=self.COLORS["accent_blue"],
            hover_color="#1B6FD4",
            corner_radius=8,
            command=self._on_send,
        )
        self.send_btn.pack(side="right", padx=(8, 0))

        ctk.CTkButton(
            btn_frame,
            text="Clear",
            width=80, height=34,
            font=ctk.CTkFont(size=13),
            fg_color="transparent",
            border_color=self.COLORS["border"],
            border_width=1,
            text_color=self.COLORS["text_secondary"],
            hover_color=self.COLORS["bg_card"],
            corner_radius=8,
            command=self._clear_session,
        ).pack(side="right")

    # ── Info Panel (right) ────────────────────────────────────────────────

    def _build_info_panel(self, parent):
        panel = ctk.CTkScrollableFrame(
            parent,
            fg_color=self.COLORS["bg_primary"],
            scrollbar_button_color=self.COLORS["border"],
            corner_radius=0,
        )
        panel.grid(row=0, column=1, sticky="nsew", padx=0, pady=0)
        panel.grid_columnconfigure(0, weight=1)

        def section_label(text):
            ctk.CTkLabel(
                panel,
                text=text,
                font=ctk.CTkFont(size=11, weight="bold"),
                text_color=self.COLORS["text_secondary"],
                anchor="w",
            ).pack(fill="x", padx=14, pady=(16, 4))

        # ── Confirmed symptoms ────────────────────────────
        section_label("CONFIRMED SYMPTOMS")
        self.sym_frame = ctk.CTkFrame(
            panel,
            fg_color=self.COLORS["bg_card"],
            corner_radius=10,
        )
        self.sym_frame.pack(fill="x", padx=12, pady=0)
        self.sym_placeholder = ctk.CTkLabel(
            self.sym_frame,
            text="No symptoms detected yet.",
            font=ctk.CTkFont(size=12),
            text_color=self.COLORS["text_secondary"],
        )
        self.sym_placeholder.pack(padx=12, pady=12)

        # ── Predictions ───────────────────────────────────
        section_label("PREDICTED CONDITIONS")
        self.pred_frame = ctk.CTkFrame(
            panel,
            fg_color=self.COLORS["bg_card"],
            corner_radius=10,
        )
        self.pred_frame.pack(fill="x", padx=12, pady=0)
        self.pred_placeholder = ctk.CTkLabel(
            self.pred_frame,
            text="Awaiting symptoms …",
            font=ctk.CTkFont(size=12),
            text_color=self.COLORS["text_secondary"],
        )
        self.pred_placeholder.pack(padx=12, pady=12)

        # ── Suggestions ───────────────────────────────────
        section_label("YOU MAY ALSO HAVE")
        self.sug_frame = ctk.CTkFrame(
            panel,
            fg_color=self.COLORS["bg_card"],
            corner_radius=10,
        )
        self.sug_frame.pack(fill="x", padx=12, pady=0)
        self.sug_placeholder = ctk.CTkLabel(
            self.sug_frame,
            text="No suggestions yet.",
            font=ctk.CTkFont(size=12),
            text_color=self.COLORS["text_secondary"],
        )
        self.sug_placeholder.pack(padx=12, pady=12)

        # ── Confidence Scores ─────────────────────────────
        section_label("CONFIDENCE SCORES")
        self.conf_frame = ctk.CTkFrame(
            panel,
            fg_color=self.COLORS["bg_card"],
            corner_radius=10,
        )
        self.conf_frame.pack(fill="x", padx=12, pady=(0, 16))
        self.conf_placeholder = ctk.CTkLabel(
            self.conf_frame,
            text="No scores yet.",
            font=ctk.CTkFont(size=12),
            text_color=self.COLORS["text_secondary"],
        )
        self.conf_placeholder.pack(padx=12, pady=12)

    # ── Footer ────────────────────────────────────────────────────────────

    def _build_footer(self):
        footer = ctk.CTkFrame(
            self,
            fg_color=self.COLORS["bg_secondary"],
            corner_radius=0, height=28,
        )
        footer.grid(row=2, column=0, sticky="ew")
        footer.grid_propagate(False)
        ctk.CTkLabel(
            footer,
            text="⚕  For educational purposes only — not a substitute for professional medical advice.",
            font=ctk.CTkFont(size=11),
            text_color=self.COLORS["text_secondary"],
        ).place(relx=0.5, rely=0.5, anchor="center")

    # ── Message Rendering ─────────────────────────────────────────────────

    def _post_user_message(self, text: str):
        outer = ctk.CTkFrame(self.chat_scroll, fg_color="transparent")
        outer.pack(fill="x", padx=12, pady=(6, 2))

        ctk.CTkLabel(
            outer,
            text="You",
            font=ctk.CTkFont(size=11, weight="bold"),
            text_color=self.COLORS["accent_blue"],
        ).pack(anchor="e")

        bubble = ctk.CTkFrame(
            outer,
            fg_color=self.COLORS["user_bubble"],
            corner_radius=12,
        )
        bubble.pack(anchor="e", padx=(80, 0))
        ctk.CTkLabel(
            bubble,
            text=text,
            font=ctk.CTkFont(size=13),
            text_color=self.COLORS["text_primary"],
            wraplength=420,
            justify="left",
        ).pack(padx=14, pady=10)

    def _post_bot_message(self, text: str):
        outer = ctk.CTkFrame(self.chat_scroll, fg_color="transparent")
        outer.pack(fill="x", padx=12, pady=(6, 2))

        ctk.CTkLabel(
            outer,
            text="MedAssist",
            font=ctk.CTkFont(size=11, weight="bold"),
            text_color=self.COLORS["accent_teal"],
        ).pack(anchor="w")

        bubble = ctk.CTkFrame(
            outer,
            fg_color=self.COLORS["bot_bubble"],
            corner_radius=12,
            border_color=self.COLORS["border"],
            border_width=1,
        )
        bubble.pack(anchor="w", padx=(0, 80))
        ctk.CTkLabel(
            bubble,
            text=text,
            font=ctk.CTkFont(size=13),
            text_color=self.COLORS["text_primary"],
            wraplength=420,
            justify="left",
        ).pack(padx=14, pady=10)

        self._scroll_to_bottom()

    def _scroll_to_bottom(self):
        self.after(100, lambda: self.chat_scroll._parent_canvas.yview_moveto(1.0))

    # ── Event Handlers ────────────────────────────────────────────────────

    def _on_enter(self, event):
        if not event.state & 0x1:   # Shift not held
            self._on_send()
            return "break"

    def _on_send(self):
        text = self.input_box.get("1.0", "end").strip()
        if not text:
            return
        self.input_box.delete("1.0", "end")
        self._post_user_message(text)
        self.send_btn.configure(state="disabled", text="Analysing …")
        threading.Thread(target=self._run_analysis, args=(text,), daemon=True).start()

    def _run_analysis(self, text: str):
        """Runs engine in background thread, updates UI on main thread."""
        try:
            confirmed, scores = self.engine.extract_symptoms(text, return_scores=True)
            new_syms = [s for s in confirmed if s not in self.confirmed_syms]
            self.confirmed_syms.extend(new_syms)

            predictions   = self.engine.predict_top_two(self.confirmed_syms)
            suggestions   = self.engine.get_recommendations(self.confirmed_syms)
            self.after(0, self._update_ui, new_syms, predictions, suggestions, scores)

        except Exception as exc:
            self.after(0, self._post_bot_message, f"⚠ Error: {exc}")
            self.after(0, lambda: self.send_btn.configure(state="normal", text="Send  ↵"))

    def _update_ui(self, new_syms, predictions, suggestions, scores):
        """Must be called from the main thread."""
        # ── Bot reply ─────────────────────────────────────
        if not new_syms:
            if self.confirmed_syms:
                self._post_bot_message(
                    "I didn't detect additional symptoms in that message. "
                    "Could you describe any other discomfort you're feeling?"
                )
            else:
                self._post_bot_message(
                    "I couldn't identify any medical symptoms in that message. "
                    "Please try describing your discomfort in more detail."
                )
        else:
            sym_list = ", ".join(s.replace("_", " ").title() for s in new_syms)
            msg = f"I detected: {sym_list}.\n"
            if predictions:
                top = predictions[0]
                msg += f"\nBased on all symptoms so far, the most likely condition is "
                msg += f"{top[0].replace('_',' ').title()} ({top[1]:.1f}% confidence)."
            if suggestions:
                msg += f"\n\nAre you also experiencing: {', '.join(suggestions)}?"
            self._post_bot_message(msg)

        # ── Right panel updates ───────────────────────────
        self._refresh_symptoms_panel()
        self._refresh_predictions_panel(predictions)
        self._refresh_suggestions_panel(suggestions)
        self._refresh_confidence_panel(scores)

        self.send_btn.configure(state="normal", text="Send  ↵")

    # ── Right Panel Refresh Methods ───────────────────────────────────────

    def _clear_frame(self, frame):
        for w in frame.winfo_children():
            w.destroy()

    def _refresh_symptoms_panel(self):
        self._clear_frame(self.sym_frame)
        if not self.confirmed_syms:
            ctk.CTkLabel(
                self.sym_frame,
                text="No symptoms detected yet.",
                font=ctk.CTkFont(size=12),
                text_color=self.COLORS["text_secondary"],
            ).pack(padx=12, pady=12)
            return

        wrap = ctk.CTkFrame(self.sym_frame, fg_color="transparent")
        wrap.pack(fill="x", padx=10, pady=10)

        row_frame = None
        for i, sym in enumerate(self.confirmed_syms):
            if i % 2 == 0:
                row_frame = ctk.CTkFrame(wrap, fg_color="transparent")
                row_frame.pack(fill="x", pady=2)
            tag = ctk.CTkFrame(
                row_frame,
                fg_color=self.COLORS["tag_sym"],
                corner_radius=6,
            )
            tag.pack(side="left", padx=(0, 6))
            ctk.CTkLabel(
                tag,
                text=f"● {sym.replace('_', ' ').title()}",
                font=ctk.CTkFont(size=12),
                text_color="#58A6FF",
            ).pack(padx=8, pady=4)

    def _refresh_predictions_panel(self, predictions):
        self._clear_frame(self.pred_frame)
        if not predictions:
            ctk.CTkLabel(
                self.pred_frame,
                text="Awaiting symptoms …",
                font=ctk.CTkFont(size=12),
                text_color=self.COLORS["text_secondary"],
            ).pack(padx=12, pady=12)
            return

        colors_fg = ["#1A3A2A", "#3A2A0A"]
        text_cols = ["#3FB950", "#D29922"]

        for rank, (disease, conf) in enumerate(predictions):
            card = ctk.CTkFrame(
                self.pred_frame,
                fg_color=colors_fg[rank],
                corner_radius=8,
            )
            card.pack(fill="x", padx=10, pady=(8, 4))

            top_row = ctk.CTkFrame(card, fg_color="transparent")
            top_row.pack(fill="x", padx=10, pady=(8, 2))

            label = f"#{rank+1}  {disease.replace('_', ' ').title()}"
            ctk.CTkLabel(
                top_row,
                text=label,
                font=ctk.CTkFont(size=13, weight="bold"),
                text_color=text_cols[rank],
            ).pack(side="left")

            ctk.CTkLabel(
                top_row,
                text=f"{conf:.1f}%",
                font=ctk.CTkFont(size=13, weight="bold"),
                text_color=text_cols[rank],
            ).pack(side="right")

            # Confidence bar
            bar_bg = ctk.CTkFrame(card, fg_color=self.COLORS["bg_primary"], height=6, corner_radius=3)
            bar_bg.pack(fill="x", padx=10, pady=(2, 8))
            bar_bg.pack_propagate(False)

            fill_w = max(4, int(conf / 100 * 260))
            ctk.CTkFrame(
                bar_bg,
                fg_color=text_cols[rank],
                width=fill_w, height=6,
                corner_radius=3,
            ).pack(side="left")

    def _refresh_suggestions_panel(self, suggestions):
        self._clear_frame(self.sug_frame)
        if not suggestions:
            ctk.CTkLabel(
                self.sug_frame,
                text="No suggestions yet.",
                font=ctk.CTkFont(size=12),
                text_color=self.COLORS["text_secondary"],
            ).pack(padx=12, pady=12)
            return

        wrap = ctk.CTkFrame(self.sug_frame, fg_color="transparent")
        wrap.pack(fill="x", padx=10, pady=10)
        for sym in suggestions:
            row = ctk.CTkFrame(wrap, fg_color="transparent")
            row.pack(fill="x", pady=1)
            ctk.CTkLabel(
                row,
                text=f"◇  {sym.title()}",
                font=ctk.CTkFont(size=12),
                text_color=self.COLORS["text_secondary"],
                anchor="w",
            ).pack(fill="x", padx=4)

    def _refresh_confidence_panel(self, scores: dict):
        self._clear_frame(self.conf_frame)
        if not scores:
            ctk.CTkLabel(
                self.conf_frame,
                text="No scores yet.",
                font=ctk.CTkFont(size=12),
                text_color=self.COLORS["text_secondary"],
            ).pack(padx=12, pady=12)
            return

        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        wrap = ctk.CTkFrame(self.conf_frame, fg_color="transparent")
        wrap.pack(fill="x", padx=10, pady=8)

        for sym, conf in sorted_scores[:8]:   # show top 8
            row = ctk.CTkFrame(wrap, fg_color="transparent")
            row.pack(fill="x", pady=3)

            label_text = sym.replace("_", " ").title()
            ctk.CTkLabel(
                row,
                text=label_text,
                font=ctk.CTkFont(size=11),
                text_color=self.COLORS["text_primary"],
                width=150,
                anchor="w",
            ).pack(side="left")

            # Color by confidence tier
            if conf >= 0.8:
                bar_color = self.COLORS["accent_teal"]
            elif conf >= 0.6:
                bar_color = self.COLORS["accent_blue"]
            elif conf >= 0.4:
                bar_color = self.COLORS["accent_amber"]
            else:
                bar_color = self.COLORS["accent_red"]

            bar_bg = ctk.CTkFrame(
                row, fg_color=self.COLORS["bg_primary"],
                height=8, corner_radius=4,
                width=120,
            )
            bar_bg.pack(side="left", padx=(6, 4))
            bar_bg.pack_propagate(False)

            fill_w = max(4, int(conf * 116))
            ctk.CTkFrame(
                bar_bg,
                fg_color=bar_color,
                width=fill_w, height=8,
                corner_radius=4,
            ).pack(side="left")

            ctk.CTkLabel(
                row,
                text=f"{conf:.0%}",
                font=ctk.CTkFont(size=11),
                text_color=bar_color,
                width=36,
            ).pack(side="left")

    # ── Session Management ────────────────────────────────────────────────

    def _clear_session(self):
        self.confirmed_syms = []
        for w in self.chat_scroll.winfo_children():
            w.destroy()
        self._clear_frame(self.sym_frame)
        self._clear_frame(self.pred_frame)
        self._clear_frame(self.sug_frame)
        self._clear_frame(self.conf_frame)
        self._refresh_symptoms_panel()
        self._refresh_predictions_panel([])
        self._refresh_suggestions_panel([])
        self._refresh_confidence_panel({})
        self._post_bot_message(
            "Session cleared. Please describe your symptoms again."
        )


print("✅  MedicalGUI class defined.")

✅  MedicalGUI class defined.


## Cell 6 — Unit Tests (no GUI required)

In [13]:
# ─────────────────────────────────────────────
#  CELL 6 · Unit Tests  (runs without CSV)
# ─────────────────────────────────────────────
import types, unittest

class _StubEngine:
    """Minimal stub to test NLP methods without a real dataset."""
    symptoms_list  = ["high_fever", "cough", "headache", "chills", "diarrhoea"]
    clean_symptoms = [s.replace("_", " ") for s in symptoms_list]
    cooccurrence_graph = {
        "high_fever": [("chills", 0.6), ("headache", 0.3)],
    }

    def __init__(self):
        from spellchecker import SpellChecker
        self.spell = SpellChecker()
        self._protected_vocab = MEDICAL_DOMAIN_WORDS.copy()

    # Bind methods from MedicalEngine onto stub
    correct_spelling       = MedicalEngine.correct_spelling
    is_noise               = MedicalEngine.is_noise
    _layer1_dictionary_match = MedicalEngine._layer1_dictionary_match
    _is_negated            = MedicalEngine._is_negated
    _is_negated_heuristic  = MedicalEngine._is_negated_heuristic
    _resolve_conflicts     = MedicalEngine._resolve_conflicts
    _apply_context_boost   = MedicalEngine._apply_context_boost
    _compute_scores        = MedicalEngine._compute_scores


class TestPipeline(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.eng = _StubEngine()

    # ── Spelling correction ───────────────────────
    def test_spell_corrects_typo(self):
        out = self.eng.correct_spelling("I have a fevr")
        self.assertIn("fever", out)

    def test_spell_preserves_medical(self):
        out = self.eng.correct_spelling("nausea and vertigo")
        self.assertIn("nausea", out)
        self.assertIn("vertigo", out)

    def test_spell_preserves_short(self):
        out = self.eng.correct_spelling("I am ill")
        self.assertIn("ill", out)

    # ── Noise filter ──────────────────────────────
    def test_noise_rejects_fine(self):
        self.assertTrue(self.eng.is_noise("i am fine"))

    def test_noise_passes_symptom(self):
        self.assertFalse(self.eng.is_noise("I have a high fever and headache"))

    # ── Dictionary match ──────────────────────────
    def test_dict_match_exact(self):
        doc    = nlp("i have high fever")
        masked = set()
        result = self.eng._layer1_dictionary_match(doc, masked)
        syms   = [r[0] for r in result]
        self.assertIn("high_fever", syms)

    def test_dict_mask_prevents_overlap(self):
        doc    = nlp("high fever")
        masked = set()
        result = self.eng._layer1_dictionary_match(doc, masked)
        # Both tokens should be masked
        self.assertGreaterEqual(len(masked), 2)

    # ── Negation ──────────────────────────────────
    def test_negation_detected(self):
        doc = nlp("i have no fever")
        # 'fever' is at index 3
        self.assertTrue(self.eng._is_negated(doc, 3, 4))

    def test_negation_not_cross_sentence(self):
        doc = nlp("no fever. I do have a cough.")
        # 'cough' should NOT be negated
        cough_idx = next(
            (i for i, t in enumerate(doc) if t.text == "cough"), None
        )
        if cough_idx is not None:
            self.assertFalse(self.eng._is_negated(doc, cough_idx, cough_idx + 1))

    # ── Conflict resolution ───────────────────────
    def test_conflict_removes_lower(self):
        scored = {"chills": 0.9, "absence_of_chills": 0.4}
        result = self.eng._resolve_conflicts(scored)
        self.assertIn("chills", result)
        self.assertNotIn("absence_of_chills", result)

    # ── Context boost ─────────────────────────────
    def test_boost_increases_score(self):
        scored = {"high_fever": 0.8, "chills": 0.5}
        boosted = self.eng._apply_context_boost(scored)
        self.assertGreaterEqual(boosted["chills"], 0.5)

    # ── Scoring ───────────────────────────────────
    def test_multi_layer_bonus(self):
        l1 = [("high_fever", 1.0, 0, 2)]
        l2 = [("high_fever", 0.85)]
        l3 = []
        scored = self.eng._compute_scores(l1, l2, l3)
        # Should exceed base of 1.0 * 1.0 = 1.0 due to cap, but agreement bonus applied
        self.assertIn("high_fever", scored)
        self.assertLessEqual(scored["high_fever"], 1.0)


# Run
loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestPipeline)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

if result.wasSuccessful():
    print("\n✅  All unit tests passed.")
else:
    print(f"\n⚠  {len(result.failures)} failure(s), {len(result.errors)} error(s).")

test_boost_increases_score (__main__.TestPipeline.test_boost_increases_score) ... ok
test_conflict_removes_lower (__main__.TestPipeline.test_conflict_removes_lower) ... ok
test_dict_mask_prevents_overlap (__main__.TestPipeline.test_dict_mask_prevents_overlap) ... ok
test_dict_match_exact (__main__.TestPipeline.test_dict_match_exact) ... ok
test_multi_layer_bonus (__main__.TestPipeline.test_multi_layer_bonus) ... ok
test_negation_detected (__main__.TestPipeline.test_negation_detected) ... ok
test_negation_not_cross_sentence (__main__.TestPipeline.test_negation_not_cross_sentence) ... ok
test_noise_passes_symptom (__main__.TestPipeline.test_noise_passes_symptom) ... ok
test_noise_rejects_fine (__main__.TestPipeline.test_noise_rejects_fine) ... ok
test_spell_corrects_typo (__main__.TestPipeline.test_spell_corrects_typo) ... FAIL
test_spell_preserves_medical (__main__.TestPipeline.test_spell_preserves_medical) ... ok
test_spell_preserves_short (__main__.TestPipeline.test_spell_preserves_sh


⚠  1 failure(s), 0 error(s).


## Cell 7 — Launch Application

> **Before running:** set `CSV_PATH` and `TEST_CSV_PATH` to your actual dataset files.
>
> Expected CSV format:
> - Training CSV — one column per symptom (0/1) + a `prognosis` column with disease names
> - Test CSV — same structure as training CSV
>
> Example column names: `itching`, `skin_rash`, `high_fever`, …, `prognosis`

In [14]:
# ─────────────────────────────────────────────
#  CELL 7 · Launch
# ─────────────────────────────────────────────

CSV_PATH      = "Training.csv"    # ← change to your training CSV path
TEST_CSV_PATH = "Testing.csv"     # ← change to your test CSV path (optional)

# ── Instantiate engine ───────────────────────
engine = MedicalEngine(CSV_PATH)

# ── Optional: validate accuracy before launch ─
import os
if os.path.exists(TEST_CSV_PATH):
    engine.run_system_test(TEST_CSV_PATH)

# ── Quick CLI smoke-test ──────────────────────
test_inputs = [
    "I have a high fever and severe headache",
    "No fever but I have a sore throat and cough",
    "I feel fine",
    "I am experiencing fatigue, joint pain, and slight nausea",
    "I have shortness of breath and chest pain",
]

print("\n─── Smoke Test ───")
for inp in test_inputs:
    syms, scores = engine.extract_symptoms(inp, return_scores=True)
    preds = engine.predict_top_two(syms)
    print(f"\nInput : {inp}")
    print(f"Syms  : {syms}")
    print(f"Scores: { {k: f'{v:.2f}' for k,v in sorted(scores.items(), key=lambda x:-x[1])[:4]} }")
    if preds:
        print(f"Pred  : {preds[0][0]}  ({preds[0][1]:.1f}%)")

# ── Launch GUI ───────────────────────────────
print("\n🚀  Launching MedAssist GUI …")
app = MedicalGUI(engine)
app.mainloop()

⚙  Loading dataset …
⚙  Training RandomForest …


KeyboardInterrupt: 